# Olist Review-Analysis Pipeline — MLWB entry point

This notebook is the entry point the Celonis Action Flow executes. The first code
cell is tagged `parameters` (View → Cell Toolbar → Tags) so the Action Flow can
overwrite its values at runtime. All real logic lives in `src/`.

**Before committing:** Kernel → Restart & Clear Output (or
`jupyter nbconvert --clear-output --inplace pipeline.ipynb`) to avoid leaking data.

In [ ]:
# Cell tagged "parameters" — the Action Flow overwrites these at runtime.
input_filename = None   # CSV delivered into input-data/; None -> local sample
max_rows = None         # optional cap for cheap runs

In [ ]:
# Main execution
import os
import sys
from dotenv import load_dotenv

load_dotenv()

# In a notebook there is no __file__, so add the current working directory to the path.
sys.path.insert(0, os.path.abspath(""))

from src.pipeline import run
from src.ingestion import push_to_celonis

result = run(input_filename=input_filename, max_rows=max_rows)
print(f"Analysed {len(result)} reviews")
result.head()

In [ ]:
# Persist results. Honors ingestion.write_back in config.yaml:
#   false -> writes output/review_insights.parquet locally (S3 skipped)
#   true  -> pushes to the Celonis data pool via the S3 ingestion API
destination = push_to_celonis(result)
print(f"Done: {destination}")